In [1]:
import os
import cv2
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt

from tensorflow.keras.models import load_model

2026-08-08 16:02:28.166864: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [2]:
model = load_model("EfficientNetB0.keras")

print("Model loaded.")

Model loaded.


In [3]:
# Backbone
base_model = model.get_layer("efficientnetb0")

# Last convolution layer
last_conv_layer = base_model.get_layer("top_conv")

feature_extractor = tf.keras.Model(
    inputs=base_model.input,
    outputs=last_conv_layer.output
)

print("Feature extractor created.")

Feature extractor created.


In [4]:
classifier_input = tf.keras.Input(
    shape=last_conv_layer.output.shape[1:]
)

x = classifier_input

x = model.get_layer("global_average_pooling2d")(x)
x = model.get_layer("dropout")(x)
x = model.get_layer("dense")(x)

classifier_model = tf.keras.Model(
    classifier_input,
    x
)

print("Classifier model created.")

Classifier model created.


In [5]:
print(feature_extractor.output_shape)
print(classifier_model.output_shape)

(None, 7, 7, 1280)
(None, 20)


In [6]:
save_dir = "Figure/Figure10"

os.makedirs(save_dir, exist_ok=True)


def plot_gradcam_from_path(image_path,
                           save_name,
                           alpha=0.45):

    # -------------------------
    # Read image
    # -------------------------

    image = cv2.imread(image_path)

    image = cv2.cvtColor(
        image,
        cv2.COLOR_BGR2RGB
    )

    image = cv2.resize(image, (224,224))

    image = image.astype("float32")

    image_batch = np.expand_dims(image, axis=0)

    # -------------------------
    # Prediction
    # -------------------------

    pred = model.predict(
        image_batch,
        verbose=0
    )

    pred_index = np.argmax(pred)
    
    # Debug
    print("--------------------------------")
    print("Image:", image_path)
    print("Prediction index:", pred_index)
    print("Confidence:", np.max(pred))

    # -------------------------
    # Grad-CAM
    # -------------------------

    with tf.GradientTape() as tape:

        feature_maps = feature_extractor(image_batch)

        tape.watch(feature_maps)

        predictions = classifier_model(feature_maps)

        loss = predictions[:, pred_index]

    grads = tape.gradient(
        loss,
        feature_maps
    )

    pooled_grads = tf.reduce_mean(
        grads,
        axis=(0,1,2)
    )

    feature_maps = feature_maps[0]

    heatmap = feature_maps @ pooled_grads[..., tf.newaxis]

    heatmap = tf.squeeze(heatmap)

    heatmap = tf.maximum(heatmap,0)

    heatmap /= tf.reduce_max(heatmap)+1e-8

    heatmap = heatmap.numpy()

    # -------------------------
    # Overlay
    # -------------------------

    heatmap = cv2.resize(
        heatmap,
        (224,224)
    )

    heatmap_uint8 = np.uint8(
        255*heatmap
    )

    heatmap_color = cv2.applyColorMap(
        heatmap_uint8,
        cv2.COLORMAP_JET
    )

    heatmap_color = cv2.cvtColor(
        heatmap_color,
        cv2.COLOR_BGR2RGB
    )

    overlay = cv2.addWeighted(
        image.astype("uint8"),
        1-alpha,
        heatmap_color,
        alpha,
        0
    )

    # -------------------------
    # Save
    # -------------------------

    plt.imsave(
        os.path.join(save_dir, save_name),
        overlay
    )

    print(save_name, "saved.")

In [8]:
plot_gradcam_from_path(
    "test/ARCIGERA FLOWER MOTH/1.jpg",
    "1_Original_CAM.png"
)

plot_gradcam_from_path(
    "test_dark/ARCIGERA FLOWER MOTH/1.jpg",
    "1_Low-light_CAM.png"
)

--------------------------------
Image: test/ARCIGERA FLOWER MOTH/1.jpg
Prediction index: 11
Confidence: 0.5510465
1_Original_CAM.png saved.
--------------------------------
Image: test_dark/ARCIGERA FLOWER MOTH/1.jpg
Prediction index: 0
Confidence: 0.7946015
1_Low-light_CAM.png saved.
